In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error,r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [3]:
df=pd.read_csv('data/stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [4]:
x=df.drop(['math_score'],axis=1)
y=df.math_score

In [5]:
num_cols=list(x.select_dtypes(include='number'))
cat_cols=list(x.select_dtypes(exclude='number'))

In [6]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=1)

In [7]:
num_transformer=StandardScaler()
cat_transformer=OneHotEncoder(drop='first')

In [8]:
preprocessor=ColumnTransformer(transformers=[
    ('one_hot_encoder',cat_transformer,cat_cols),
    ('standrad_scaler',num_transformer,num_cols)
],remainder='passthrough')

In [9]:
x_train=preprocessor.fit_transform(x_train)
x_test=preprocessor.transform(x_test)

In [10]:
def evaluate_model(model,x,y):
    y_pred=model.predict(x)
    mse=mean_squared_error(y,y_pred)
    r2=r2_score(y,y_pred)
    return mse,r2

In [15]:
models={
    'linear regression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'ElasticNet':ElasticNet(),
    'KNeighborsRegressor':KNeighborsRegressor(),
    'SVR':SVR(),
    'DecisionTreeRegressor':DecisionTreeRegressor(),
    'RandomForestRegressor':RandomForestRegressor(),
    'AdaBoostRegressor':AdaBoostRegressor(),
    'GradientBoostingRegressor':GradientBoostingRegressor(),
    'XGBRegressor':XGBRegressor(),
    'CatBoostRegressor':CatBoostRegressor(verbose=0,train_dir='../logs/catboost_info/')
}

mse_train_list=list()
mse_test_list=list()
r2_train_list=list()
r2_test_list=list()
model_list=list()

for i in range(len(list(models))):
    model_name=list(models.keys())[i]
    model=list(models.values())[i]
    model.fit(x_train,y_train)

    model_train_mse,model_train_r2=evaluate_model(model,x_train,y_train)
    model_test_mse,model_test_r2=evaluate_model(model,x_test,y_test)

    model_list.append(model_name)
    mse_train_list.append(model_train_mse)
    mse_test_list.append(model_test_mse)
    r2_train_list.append(model_train_r2)
    r2_test_list.append(model_test_r2)

    print('='*35)
    print(f'{model_name} learner metrics:')
    print('-'*20)
    print('training:')
    print(f"mse : {model_train_mse}")
    print(f"r2_score :{model_train_r2}")

    print('-'*20)
    print('testing:')
    print(f"mse : {model_test_mse}")
    print(f"r2_score :{model_test_r2}")

linear regression learner metrics:
--------------------
training:
mse : 29.081964579424557
r2_score :0.8723601852982044
--------------------
testing:
mse : 26.569132426804828
r2_score :0.8867604134091436
Lasso learner metrics:
--------------------
training:
mse : 44.090410670631215
r2_score :0.8064885942366142
--------------------
testing:
mse : 40.36287577008129
r2_score :0.8279704699273176
Ridge learner metrics:
--------------------
training:
mse : 29.087496670656577
r2_score :0.8723359051262841
--------------------
testing:
mse : 26.535584766305455
r2_score :0.8869033959930313
ElasticNet learner metrics:
--------------------
training:
mse : 68.1514932958092
r2_score :0.700885270244701
--------------------
testing:
mse : 66.39950149131177
r2_score :0.7170004658816279
KNeighborsRegressor learner metrics:
--------------------
training:
mse : 34.15013333333333
r2_score :0.8501161543332528
--------------------
testing:
mse : 45.91968000000001
r2_score :0.804286963682022
SVR learner metri

In [16]:
model_perf=pd.DataFrame({
    'model':model_list,
    'train_mse':mse_train_list,
    'test_mse':mse_test_list,
    'train_r2_score':r2_train_list,
    'test_r2_score':r2_test_list
})
model_perf

,model,train_mse,test_mse,train_r2_score,test_r2_score
0,linear regression,29.081965,26.569132,0.872360,0.886760
1,Lasso,44.090411,40.362876,0.806489,0.827970
2,Ridge,29.087497,26.535585,0.872336,0.886903
3,ElasticNet,68.151493,66.399501,0.700885,0.717000
4,KNeighborsRegressor,34.150133,45.919680,0.850116,0.804287
5,SVR,48.004471,55.564254,0.789310,0.763181
6,DecisionTreeRegressor,0.019333,64.676000,0.999915,0.724346
7,RandomForestRegressor,5.456655,34.779736,0.976051,0.851766
8,AdaBoostRegressor,32.406388,36.939523,0.857769,0.842561
9,GradientBoostingRegressor,21.626630,29.433807,0.905081,0.874551
